**Reprodukcje oryginalnych modeli:** użyj [native_worldmodels_colab.ipynb](https://colab.research.google.com/github/twojtys137/lpworldmodel/blob/experiment/theory-colab-1800/notebooks/native_worldmodels_colab.ipynb). Ten notebook zachowuje eksperymenty generatorów i kontroli RDM.

# LpWM: eksperymenty teorii — budżet 1800 CU

Domyślny etap `pilot` używa czterech istniejących checkpointów `fair_lpwm_v1_screen_*_seed0`.
Sprawdza predykcję, rzeczywistą lokalność, kolejność działań oraz projekcję kosztu celu.
Pełne treningi są osobnymi etapami. Wyniki zapisują się na MyDrive.

**Stan w repo:** testy CPU i mały test całego przepływu są wykonane. Wyniki PushT na wytrenowanych modelach wymagają uruchomienia tego notebooka.
**Zakres B:** projekcja kosztu końcowego; pełny predyktor nadal pracuje. To nie jest jeszcze skompresowana dynamika.

Uruchom komórki przygotowania; po pobraniu danych wybierz GPU. Wpisz aktualne CU/h z panelu zasobów Colaba.
Nie instalujemy ani nie aktualizujemy PyTorcha/CUDA dostarczonych przez Colaba.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
REPOSITORY = 'https://github.com/twojtys137/lpworldmodel.git'
REPO_REF = 'experiment/theory-colab-1800'
REPO = Path('/content/lpwm-theory')
if not REPO.exists():
    subprocess.run(['git','clone','--branch',REPO_REF,'--single-branch',REPOSITORY,str(REPO)],check=True)
if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('Katalog repo zawiera lokalne zmiany. Zachowaj je lub wybierz nowy REPO.')
subprocess.run(['git','-C',str(REPO),'fetch','origin',REPO_REF],check=True)
COMMIT = subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD'],text=True).strip()
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
os.chdir(REPO)
print('Commit tej sesji:', COMMIT)

In [ ]:
dependencies = [
    'accelerate>=0.26,<2','hydra-core>=1.3,<1.4','omegaconf>=2.3,<3',
    'wandb>=0.13,<1','submitit>=1.5,<2','hydra-submitit-launcher>=1.2,<2',
    'gym==0.26.2','pymunk==6.11.1','moviepy<2',
    'einops','decord','pygame','shapely','scikit-image','tensorboardX',
    'requests','tqdm','psutil','pytest','scipy','opencv-python-headless','matplotlib',
]
subprocess.run([sys.executable,'-m','pip','install','-q',*dependencies],check=True)
subprocess.run([sys.executable,'-m','pytest','-q','tests'],check=True)
subprocess.run([sys.executable,'-c','import train, plan; import pymunk; assert hasattr(pymunk.Space(), "add_collision_handler")'],check=True)

## Dane i zapis

Dane pracują na lokalnym dysku `/content/lpwm-data`; checkpointy i logi na MyDrive.
Pobranie można wykonać na CPU. Po zmianie typu runtime sprawdź dostępność danych ponownie.
Klucz W&B jest pobierany z Colab Secrets. Nie trafia do plików ani parametrów komend.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
RESULTS = Path('/content/drive/MyDrive/lpwm-sparse-generator')
DATA = Path('/content/lpwm-data')
CAMPAIGN_ID = 'theory_v1'
OUTPUT = RESULTS / 'theory_experiments' / CAMPAIGN_ID
OUTPUT.mkdir(parents=True,exist_ok=True)
os.environ.update(DATASET_DIR=str(DATA),CKPT_BASE=str(RESULTS),ENV_NAME='pusht',
                  SDL_VIDEODRIVER='dummy',WANDB_DIR=str(RESULTS),
                  WANDB_ENTITY='twojtys137-tw',WANDB_PROJECT='lpwm-sparse-generator')
ALLOW_WANDB_OFFLINE = False
try:
    key = userdata.get('WANDB_API_KEY')
except Exception:
    key = None
if key:
    import wandb
    os.environ['WANDB_API_KEY'] = key
    os.environ['WANDB_MODE'] = 'online'
    if wandb.login(key=key,relogin=True) is False:
        raise RuntimeError('W&B odrzucił klucz.')
else:
    if not ALLOW_WANDB_OFFLINE:
        raise RuntimeError('Dodaj sekret WANDB_API_KEY lub jawnie ustaw ALLOW_WANDB_OFFLINE=True.')
    os.environ['WANDB_MODE']='offline'
os.environ['REQUIRE_WANDB_ONLINE']='0' if ALLOW_WANDB_OFFLINE else '1'
subprocess.run([sys.executable,'scripts/download_lpwmdatasets.py','--dataset','pusht_noise',
                '--output-dir',str(DATA)],check=True)
from datetime import datetime, timezone
session=OUTPUT/'sessions'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
session.mkdir(parents=True,exist_ok=True)
(session/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
(session/'code_commit.txt').write_text(COMMIT+'\n')
print('Wyniki:',OUTPUT)

## Wybór etapu i limit

| Etap | Limit sumy uruchomień | Cel |
|---|---:|---|
| `pilot` | 120 CU | Istniejące modele; lokalność, kolejność, cele i kontrole |
| `baseline` | 480 CU | Pełne dane, bazowe modele i planowanie |
| `factorial` | 600 CU | Dokończenie macierzy 2×2 + projekcje celu |
| `replicate` | 300 CU | Nasiona 1 i 2 dla głównego porównania |
| rezerwa | 300 CU | Nie jest uruchamiana automatycznie |

To limity czasu przeliczone podaną stawką, **nie odczyt rozliczeń Google**. Instalacja, pobieranie,
bezczynność, zmiany stawki i inne runtime'y nie wchodzą do rejestru. Sprawdzaj rzeczywiste saldo.
Jeden runtime na jeden rejestr. Przerwany runtime pozostawia konserwatywnie zarezerwowany limit.
Zadanie przerwane limitem nie jest traktowane jako ukończony trening; użyj świeżego identyfikatora próby.

`Run all` uruchomi wyłącznie wybrany etap. Domyślnie jest to pilot.

In [ ]:
PHASE = 'pilot'  # pilot | controlled_baseline | factorial | replicate; oryginały: native_worldmodels_colab.ipynb
RATE_CU_HOUR = None  # Wpisz stawkę z panelu aktywnego GPU; nie zgaduj.
TOTAL_CU = 1800
DISCONNECT_WHEN_DONE = True
PHASE_DONE = False
LEDGER = OUTPUT/'budget.json'
if PHASE not in {'pilot','baseline','controlled_baseline','factorial','replicate'}:
    raise ValueError(PHASE)
if not isinstance(RATE_CU_HOUR,(int,float)) or RATE_CU_HOUR<=0:
    raise ValueError('Wpisz bieżące CU/h z panelu zasobów Colaba w RATE_CU_HOUR.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Wybierz runtime GPU przed etapami eksperymentów.')
print(torch.cuda.get_device_name(0), 'stawka:',RATE_CU_HOUR,'CU/h')

def budget(label, cap, command, extra_env=None):
    env = os.environ.copy()
    env.update(extra_env or {})
    subprocess.run([sys.executable,'-m','experiments.budget','--ledger',str(LEDGER),
                    '--total-cu',str(TOTAL_CU),'--rate-cu-hour',str(RATE_CU_HOUR),
                    '--cap-cu',str(cap),'--label',label,'--',*map(str,command)],check=True,env=env)

def require_done(label):
    rows=json.loads(LEDGER.read_text())['runs'] if LEDGER.exists() else []
    if not any(r['label']==label and r.get('returncode')==0 for r in rows):
        raise RuntimeError('Najpierw ukończ etap: '+label)

def names(models, profile='full', seed=0):
    prefix='fair_lpwm_v1' if profile=='screen' else CAMPAIGN_ID
    return [f'{prefix}_{profile}_{model}_seed{seed}' for model in models]

FOUR=['dense_dense','dense_sparse','sparse_dense','sparse_sparse']

def checkpoint_inventory(model_names):
    for name in model_names:
        path=RESULTS/'outputs'/name/'checkpoints/model_latest.pth'
        config=path.parents[1]/'hydra.yaml'
        print(name, 'OK' if path.exists() and config.exists() else 'BRAK')
        if not path.exists() or not config.exists():
            raise FileNotFoundError(path)

def probe(model_names, root, label_prefix, cap_each=8, horizon=5):
    checkpoint_inventory(model_names)
    for name in model_names:
        budget(label_prefix+'-'+name, cap_each,
               [sys.executable,'-m','experiments.frozen_probe',
                '--run-dir',RESULTS/'outputs'/name,'--output',root/name,
                '--horizon',horizon,'--context-frames',1,'--samples',16,'--candidates',16,
                '--locality-samples',2,'--simulator-samples',4,'--require-cuda'])

def paired(model_names, folder, label, cap, basis_root=None, n=10, steps=10, max_iter=3):
    checkpoint_inventory(model_names)
    command=[sys.executable,'-m','experiments.paired_eval','--ckpt-base',RESULTS,
             '--models',*model_names,'--output',folder,'--n-evals',n,
             '--cem-steps',steps,'--max-iter',max_iter,'--rank',64]
    if basis_root is not None:
        command += ['--basis-root',basis_root]
    budget(label,cap,command)

def train_group(model_names, seeds, label, cap):
    # Existing completed runs are immutable. Do not silently promote an old partial checkpoint.
    for seed in seeds:
        for name in names(model_names,seed=seed):
            existing=RESULTS/'outputs'/name/'checkpoints/model_latest.pth'
            if existing.exists():
                require_done(label)
    budget(label,cap,['bash','scripts/benchmark_lpwm_sparse_generator_colab.sh'],
           dict(RUN='1',PROFILE='full',STAGE='train',BENCHMARK_ID=CAMPAIGN_ID,
                MODELS=' '.join(model_names),SEEDS=' '.join(map(str,seeds)),
                EPOCHS='2',N_ROLLOUT='all',NUM_PROJECTIONS='2048',
                PATCH_BATCH_SIZE='16',PAPER_BATCH_SIZE='16',RESUME='0'))

## 1. Pilot: stare checkpointy, bez treningu

A: centralne różnice wartości predyktora i przełączenia grafu; gradient treningowy routera nie jest używany.
B: baza reszt celu kontra PCA stanu i baza losowa; kalibracja i test mają rozłączne trajektorie.
C: zamiana dwóch bloków działań w modelu i symulatorze. Nie zakładamy, że przeciwna akcja odwraca ruch.

Probe startuje od jednej klatki, tak jak obecny MPC. Trzy klatki kontekstu to osobna ablacja.

Najpierw model musi dawać nietrywialny wynik względem zerowego/losowego sterowania. Mała próbka 10 celów
służy diagnostyce, nie końcowemu wnioskowaniu. Przy samych zerach badanie projekcji w MPC jest pomijane;
wyniki probe nadal pozwalają badać skalę, zapaść reprezentacji i ignorowanie akcji.

In [ ]:
if PHASE=='pilot':
    subprocess.run([sys.executable,'-m','experiments.theory_checks','--output',str(OUTPUT/'theory_cpu.json')],check=True)
    old=names(FOUR,profile='screen')
    pilot_probes=OUTPUT/'pilot_probes'
    probe(old,pilot_probes,'pilot-probe',cap_each=8)  # 32 CU
    folder=OUTPUT/'pilot_planning'
    paired(old,folder,'pilot-planning',40)  # 40 CU
    rows=json.loads((folder/'results.json').read_text())
    control=max(r['success_rate'] for r in rows if r['mode'] in {'zero_action','random_action'})
    learned=max(r['success_rate'] for r in rows if r['mode']=='plan')
    if learned>control:
        paired(names(['dense_dense','dense_sparse'],profile='screen'),OUTPUT/'pilot_projection',
               'pilot-projection',48,basis_root=pilot_probes)
    else:
        print('Brak przewagi nad kontrolą w małym pilotażu. Sprawdź probe i pełny baseline przed MPC z projekcją.')
    PHASE_DONE=True

## 2. Bazowe modele na pełnych danych

Dwie epoki, pełne dane, seed 0, 2048 projekcji regularizatora, batch 16. To kontrolowany budżetowo profil,
a nie deklaracja odtworzenia ustawień publikacji. Nazwa `lewm` w dotychczasowym launcherze oznacza tu
CLS + gęsty predyktor + Gaussian RDM; **nie dokładny SIGReg z LeWM**.
Trzecim baseline'em jest `dense_dense`, potrzebny do macierzy 2×2.

In [ ]:
if PHASE=='baseline':
    raise RuntimeError('Dla wiernej reprodukcji uruchom native_worldmodels_colab.ipynb. Dawny profil baseline jest dostępny jako controlled_baseline (Gaussian RDM, nie oryginalny LeWM).')
if PHASE=='controlled_baseline':
    models=['lewm_rdm','lpwm','dense_dense']
    train_group(models,[0],'baseline-train',300)
    paired(names(models),OUTPUT/'baseline_planning','baseline-planning',180,n=50,steps=30,max_iter=10)
    PHASE_DONE=True

## 3. Macierz 2×2 i projekcja celu

Uzupełnienie seed 0 o dense/sparse state × dense/sparse dynamics. `dense_dense` pochodzi z poprzedniego
etapu z tym samym harmonogramem. Dopasowane epoki, dane, batch i liczba aktualizacji nie oznaczają
identycznego FLOP; mierz czas, szczyt VRAM i liczbę parametrów.
Rząd 64 jest ustalony przed ewaluacją MPC. Pełny predyktor, PCA i baza losowa pozostają kontrolami. CEM buforuje kod obrazu początkowego
i ocenia 300 kandydatów w partiach po 32; zachowuje tę samą pulę akcji.
Cele MPC wykluczają trajektorie użyte zarówno do kalibracji, jak i testowania probe.

In [ ]:
if PHASE=='factorial':
    require_done('baseline-train')
    train_group(['dense_sparse','sparse_dense','sparse_sparse'],[0],'factorial-train',300)
    root=OUTPUT/'full_probes'
    probe(names(FOUR),root,'full-probe',cap_each=10)  # 40 CU
    paired(names(FOUR),OUTPUT/'factorial_planning','factorial-planning',260,
           basis_root=root,n=50,steps=30,max_iter=10)
    PHASE_DONE=True

## 4. Replikacja głównego porównania

Z góry ustalone porównanie `dense_dense` kontra `dense_sparse`; seed 0 już istnieje. Dodajemy seed 1 i 2.
To replikacja hipotezy o rzadkiej dynamice przy gęstym stanie. Replikacja kompresji celu oraz horyzontów
10/20 jest kolejnym krokiem, jeśli pomiar kosztu pokaże, że mieści się w pozostałym budżecie.
Nie traktuj dwóch wariantów na trzech seedach jak sześciu niezależnych testów.

In [ ]:
if PHASE=='replicate':
    require_done('factorial-train')
    train_group(['dense_dense','dense_sparse'],[1,2],'replicate-train',180)
    for seed in (1,2):
        paired(names(['dense_dense','dense_sparse'],seed=seed),OUTPUT/f'replicate_seed{seed}',
               f'replicate-planning-{seed}',60,n=50,steps=30,max_iter=10)
    PHASE_DONE=True

## Wyniki i zakończenie

Czytaj `results.md`, `results.json`, `compression.csv`, `progress.json` oraz `budget.json`.
Wyniki planowania mają sukcesy dla poszczególnych celów i ich pochodzenie, co pozwala na analizę parowaną.
Przedziały w `paired_deltas.json` są opisowe dla pojedynczego seeda. Końcowa analiza musi uwzględnić
klastry trajektorii i zmienność między treningami. Przy 0/10 lub 10/10 bootstrap może dać pozornie
zerową szerokość przedziału — to nie dowód braku niepewności.

Dla publikacji wymagamy przewagi w success rate na co najmniej trzech seedach, albo ustalonego wcześniej
marginesu non-inferiority i zmierzonej oszczędności. Sama aktywna liczba krawędzi nie dowodzi przyspieszenia.
Nie porównuj bezpośrednio MSE kodów o różnych rozkładach.

Po pomyślnym zakończeniu wybranego etapu runtime jest domyślnie odłączany, żeby ograniczyć koszt bezczynności.
Po błędzie sprawdź log i zatrzymaj runtime ręcznie.

In [ ]:
if LEDGER.exists():
    ledger=json.loads(LEDGER.read_text())
    spent=sum(r.get('estimated_cu',r['reserved_cu']) for r in ledger['runs'])
    print(f'Rejestr: {spent:.2f} / {TOTAL_CU} szacowanych CU; sprawdź rzeczywiste saldo Colaba.')
print('Wyniki:',OUTPUT)
if PHASE_DONE and DISCONNECT_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()